<a href="https://colab.research.google.com/github/RenaldyNaufalTA/komputasi_numerik/blob/main/Eliminasi_Gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eliminasi Gauss dengan Partial Pivoting


## 1. Konsep Dasar

Contoh SPL dengan 3 persamaan dan 3 variabel:

```
 2x +  y -  z =   8
-3x -  y + 2z = -11
-2x +  y + 2z =  -3
```

SPL ini ditulis dalam bentuk matriks **A · x = b**:
- **A** = tabel koefisien (angka di depan x, y, z)
- **x** = variabel yang dicari
- **b** = ruas kanan (angka setelah tanda `=`)

**Ide metode eliminasi Gauss** ada dua tahap:
1. **Eliminasi maju**: ubah sistem menjadi bentuk "segitiga", yaitu semua angka di bawah diagonal menjadi 0.
2. **Substitusi mundur**: hitung jawaban mulai dari baris paling bawah naik ke atas.

## 2. Persiapan: Import dan EPSILON

- `numpy` adalah pustaka Python untuk menghitung dengan array/matriks. Kita memberinya nama pendek `np`.
- `EPSILON` adalah angka yang sangat kecil (`1e-12` = 0,000000000001). Komputer menyimpan bilangan desimal secara pendekatan, sehingga hasil yang seharusnya `0` sering menjadi `0.0000000000001`. Karena itu kita tidak memeriksa `== 0`, tetapi memeriksa apakah nilainya **lebih kecil dari EPSILON**.

In [ ]:
import numpy as np

EPSILON = 1e-12   # batas "hampir nol"

## 3. Data Contoh dan Persiapan Array

```python
A = np.array(A, dtype=float)
b = np.array(b, dtype=float)
n = len(b)
```

- `np.array(...)` mengubah list Python menjadi **array NumPy** agar bisa dihitung per baris/kolom.
- `dtype=float` memastikan angkanya bilangan desimal. Tanpa ini, bilangan bulat dapat membuat hasil pembagian terpotong.
- `n = len(b)` adalah jumlah persamaan (sama dengan jumlah variabel).

In [ ]:
A = [[2, 1, -1],
     [-3, -1, 2],
     [-2, 1, 2]]
b = [8, -11, -3]

A = np.array(A, dtype=float)
b = np.array(b, dtype=float)
n = len(b)

print("Matriks A:")
print(A)
print("\nVektor b:", b)
print("Jumlah persamaan n =", n)

## 4. Matriks Augmented

**Matriks augmented** adalah matriks A dan b yang digabung dalam satu tabel `[A | b]`. Dengan begitu, setiap operasi baris otomatis berlaku untuk koefisien **dan** ruas kanan sekaligus.

```python
M = np.hstack((A, b.reshape(-1, 1)))
```

- `b.reshape(-1, 1)` mengubah b dari bentuk datar `[8, -11, -3]` menjadi **kolom** (3 baris x 1 kolom). Tanda `-1` artinya "hitung sendiri jumlah barisnya".
- `np.hstack` (*horizontal stack*) menempelkan kolom itu di sisi kanan A.

Hasilnya: M punya `n` baris dan `n+1` kolom. Kolom terakhir (indeks `n`) adalah b.

In [ ]:
print("Bentuk b setelah reshape (kolom):")
print(b.reshape(-1, 1))

M = np.hstack((A, b.reshape(-1, 1)))
print("\nMatriks augmented [A | b]:")
print(M)
print("\nUkuran M (baris, kolom):", M.shape)

## 5. Pengaturan Tampilan

```python
np.set_printoptions(precision=4, suppress=True, linewidth=120)
```

Baris ini hanya untuk mempercantik cetakan, tidak memengaruhi hasil hitungan:
- `precision=4` : tampilkan 4 angka di belakang koma
- `suppress=True` : jangan pakai notasi ilmiah (misal `1e-16`), tampilkan sebagai 0
- `linewidth=120` : lebar satu baris cetakan maksimal 120 karakter

In [ ]:
np.set_printoptions(precision=4, suppress=True, linewidth=120)
print(M / 3)   # contoh: angka desimal tampil rapi 4 digit

## 6. Tahap 1: Eliminasi Maju

```python
for k in range(n - 1):
```

Variabel `k` adalah **nomor kolom yang sedang dibereskan**, sekaligus posisi **pivot** (elemen diagonal `M[k, k]`). Perulangan berhenti di `n-2` karena kolom terakhir tidak punya elemen di bawah diagonal yang perlu dihapus. Ingat: indeks Python dimulai dari 0.

Di bawah ini kita jalankan satu putaran (`k = 0`) secara manual, dibagi dalam 4 langkah kecil.

### 6a. Partial pivoting (memilih pivot terbaik)

```python
p = k + np.argmax(np.abs(M[k:, k]))
```

Dibaca dari dalam ke luar:
- `M[k:, k]` : ambil kolom k, dari baris k sampai bawah
- `np.abs(...)` : ambil nilai mutlak (buang tanda minus)
- `np.argmax(...)` : cari **posisi** nilai terbesar (bukan nilainya)
- `k + ...` : kembalikan ke nomor baris asli, karena potongan `M[k:, ...]` dimulai dari baris k

Jadi `p` adalah nomor baris yang punya angka terbesar di kolom k.

> **Mengapa dicari yang terbesar?** Nanti kita membagi dengan pivot. Pivot yang kecil menghasilkan angka sangat besar sehingga error pembulatan komputer membengkak. Pivot besar membuat hitungan lebih stabil.

In [ ]:
k = 0
print("Kolom", k, "dari baris", k, "ke bawah :", M[k:, k])
print("Nilai mutlaknya                :", np.abs(M[k:, k]))
print("Posisi terbesar (relatif)      :", np.argmax(np.abs(M[k:, k])))

p = k + np.argmax(np.abs(M[k:, k]))
print("\nBaris pivot p =", p, "(yaitu baris R%d)" % (p + 1))
print("Nilai pivot   =", M[p, k])

### 6b. Cek pivot nol

```python
if abs(M[p, k]) < EPSILON:
    print("SPL tidak memiliki solusi tunggal (pivot = 0).")
    return None
```

Jika angka **terbesar** di kolom itu pun hampir nol, berarti seluruh kolom itu "kosong". Akibatnya SPL tidak punya solusi tunggal (bisa tidak ada solusi, atau solusinya tak hingga banyak). Fungsi langsung berhenti dan mengembalikan `None` (artinya "tidak ada hasil").

In [ ]:
if abs(M[p, k]) < EPSILON:
    print("SPL tidak memiliki solusi tunggal (pivot = 0).")
else:
    print("Pivot =", M[p, k], "-> aman, proses dilanjutkan.")

### 6c. Tukar baris

```python
if p != k:
    M[[k, p]] = M[[p, k]]
```

Jika baris terbaik (`p`) bukan baris yang sedang dikerjakan (`k`), keduanya ditukar. Penulisan `M[[k, p]] = M[[p, k]]` adalah cara NumPy menukar dua baris sekaligus tanpa variabel sementara.

Menukar urutan persamaan **tidak mengubah jawaban** SPL, hanya urutan penulisannya.

In [ ]:
print("Sebelum tukar baris:")
print(M)

if p != k:
    M[[k, p]] = M[[p, k]]
    print(f"\nTukar baris R{k+1} <-> R{p+1}")

print("\nSesudah tukar baris:")
print(M)

### 6d. Menghapus elemen di bawah pivot (operasi baris)

```python
for i in range(k + 1, n):
    m = M[i, k] / M[k, k]
    M[i, k:] = M[i, k:] - m * M[k, k:]
```

Untuk setiap baris `i` di bawah baris pivot:
1. **`m`** adalah *pengali*, yaitu perbandingan elemen yang ingin dihapus dengan pivot.
2. **Operasi baris**: `R_i <- R_i - m * R_k`. Seluruh baris pivot dikali `m`, lalu dikurangkan dari baris i.

**Mengapa elemennya menjadi nol?** Di kolom k:
`M[i,k] - (M[i,k] / M[k,k]) * M[k,k] = M[i,k] - M[i,k] = 0`

`M[i, k:]` berarti "dari kolom k sampai kolom terakhir". Kolom sebelum k sudah bernilai 0, jadi tidak perlu dihitung ulang (lebih hemat).

In [ ]:
for i in range(k + 1, n):
    m = M[i, k] / M[k, k]
    M[i, k:] = M[i, k:] - m * M[k, k:]
    print(f"R{i+1} <- R{i+1} - ({m:.4f}) * R{k+1}")

print(f"\nSetelah eliminasi kolom {k+1}:")
print(M)

### 6e. Putaran berikutnya (k = 1)

Langkah yang sama diulang untuk kolom berikutnya. Kali ini pivot dicari mulai dari baris ke-2 (karena baris 1 sudah "selesai"). Sel di bawah menggabungkan langkah 6a sampai 6d dalam satu putaran.

Setelah putaran ini, semua angka di bawah diagonal bernilai 0, sehingga matriks berbentuk **segitiga atas**.

In [ ]:
k = 1
p = k + np.argmax(np.abs(M[k:, k]))
print("Baris pivot:", f"R{p+1}", "dengan nilai", M[p, k])

if p != k:
    M[[k, p]] = M[[p, k]]
    print(f"Tukar baris R{k+1} <-> R{p+1}")

for i in range(k + 1, n):
    m = M[i, k] / M[k, k]
    M[i, k:] = M[i, k:] - m * M[k, k:]
    print(f"R{i+1} <- R{i+1} - ({m:.4f}) * R{k+1}")

print("\nMatriks segitiga atas:")
print(M)

## 7. Pengecekan Pivot Terakhir

```python
if abs(M[n - 1, n - 1]) < EPSILON:
    return None
```

Perulangan eliminasi berhenti sebelum memeriksa pivot terakhir (pojok kanan bawah matriks A). Jika nilainya nol, baris terakhir berbentuk `0 * z = sesuatu`, sehingga tidak ada solusi tunggal.

In [ ]:
if abs(M[n - 1, n - 1]) < EPSILON:
    print("SPL tidak memiliki solusi tunggal (pivot = 0).")
else:
    print("Pivot terakhir =", M[n - 1, n - 1], "-> aman.")

## 8. Tahap 2: Substitusi Mundur

Sekarang matriks sudah berbentuk segitiga atas, sehingga **baris terakhir hanya mengandung satu variabel**.

**Langkah 1: variabel terakhir**
```python
x = np.zeros(n)
x[n - 1] = M[n - 1, n] / M[n - 1, n - 1]
```
- `np.zeros(n)` menyiapkan wadah `x` berisi nol.
- Baris terakhir berbentuk `a * x_terakhir = c`, jadi `x_terakhir = c / a`.

In [ ]:
x = np.zeros(n)
x[n - 1] = M[n - 1, n] / M[n - 1, n - 1]
print("x terakhir (z) =", x[n - 1])

**Langkah 2: variabel-variabel di atasnya**

```python
for i in range(n - 2, -1, -1):
    jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
    x[i] = (M[i, n] - jumlah) / M[i, i]
```

- `range(n - 2, -1, -1)` adalah perulangan **mundur**: mulai dari baris `n-2` sampai baris `0` (parameter ketiga `-1` berarti turun satu-satu).
- Baris i berbentuk: `M[i,i] * x_i + (koefisien lain) * (x yang sudah diketahui) = b_i`.
- `jumlah` = total suku yang variabelnya **sudah diketahui**. `np.dot` mengalikan pasangan elemen lalu menjumlahkannya.
- Pindahkan `jumlah` ke ruas kanan, lalu bagi dengan koefisien pivot: `x_i = (b_i - jumlah) / M[i,i]`.

In [ ]:
for i in range(n - 2, -1, -1):
    jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
    x[i] = (M[i, n] - jumlah) / M[i, i]
    print(f"Baris {i+1}: jumlah = {jumlah:.4f}, x[{i}] = {x[i]:.4f}")

print("\nSolusi: x =", x[0], ", y =", x[1], ", z =", x[2])

## 9. Fungsi Lengkap

Semua potongan di atas digabung menjadi satu fungsi `gauss_eliminasi`. Parameter `tampil_langkah=True` membuat fungsi mencetak langkah-langkahnya. Jika diisi `False`, fungsi bekerja diam-diam dan hanya mengembalikan hasil.

Fungsi mengembalikan:
- vektor `x` (jawaban) jika SPL punya solusi tunggal
- `None` jika tidak punya solusi tunggal

In [ ]:
def gauss_eliminasi(A, b, tampil_langkah=True):
    """Menyelesaikan A.x = b dengan eliminasi Gauss.
    Mengembalikan vektor x, atau None jika tidak ada solusi tunggal."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    # Bentuk matriks augmented [A | b]
    M = np.hstack((A, b.reshape(-1, 1)))
    np.set_printoptions(precision=4, suppress=True, linewidth=120)
    if tampil_langkah:
        print("Matriks augmented awal:")
        print(M, "\n")
    # ---------- Tahap 1: Eliminasi maju ----------
    for k in range(n - 1):
        # Partial pivoting: cari baris dengan |nilai| terbesar di kolom k
        p = k + np.argmax(np.abs(M[k:, k]))
        if abs(M[p, k]) < EPSILON:
            print("SPL tidak memiliki solusi tunggal (pivot = 0).")
            return None
        if p != k:
            M[[k, p]] = M[[p, k]]  # tukar baris
            if tampil_langkah:
                print(f"Tukar baris R{k+1} <-> R{p+1}")
        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]
            M[i, k:] = M[i, k:] - m * M[k, k:]
            if tampil_langkah:
                print(f"R{i+1} <- R{i+1} - ({m:.4f}) * R{k+1}")
        if tampil_langkah:
            print(f"\nSetelah eliminasi kolom {k+1}:")
            print(M, "\n")
    if abs(M[n - 1, n - 1]) < EPSILON:
        print("SPL tidak memiliki solusi tunggal (pivot = 0).")
        return None
    # ---------- Tahap 2: Substitusi mundur ----------
    x = np.zeros(n)
    x[n - 1] = M[n - 1, n] / M[n - 1, n - 1]
    for i in range(n - 2, -1, -1):
        jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, n] - jumlah) / M[i, i]
    return x

## 10. Pengujian Fungsi

### 10a. SPL dengan solusi tunggal

Kita pakai contoh yang sama. Jawaban yang diharapkan: **x = 2, y = 3, z = -1**.

In [ ]:
A = [[2, 1, -1],
     [-3, -1, 2],
     [-2, 1, 2]]
b = [8, -11, -3]

hasil = gauss_eliminasi(A, b)
print("Solusi:", hasil)

### 10b. Verifikasi jawaban

Cara memeriksa jawaban: masukkan kembali ke persamaan. Jika benar, `A @ x` (perkalian matriks) harus sama dengan `b`. Kita juga membandingkannya dengan fungsi bawaan NumPy `np.linalg.solve`.

In [ ]:
print("A @ x      =", np.array(A) @ hasil)
print("b          =", np.array(b))
print("Cocok?     =", np.allclose(np.array(A) @ hasil, b))
print("np.linalg.solve =", np.linalg.solve(A, b))

### 10c. SPL tanpa solusi tunggal

Persamaan kedua `2x + 4y = 6` adalah kelipatan dua dari persamaan pertama `x + 2y = 3`. Artinya hanya ada satu persamaan "asli", sehingga solusinya tak hingga banyak. Fungsi harus mengembalikan `None` dan menampilkan pesan peringatan.

In [ ]:
A2 = [[1, 2],
      [2, 4]]
b2 = [3, 6]

hasil2 = gauss_eliminasi(A2, b2)
print("Hasil:", hasil2)

### 10d. Mode diam (tanpa menampilkan langkah)

Dengan `tampil_langkah=False`, fungsi hanya mengembalikan jawaban. Berguna jika kita hanya ingin hasil akhir.

In [ ]:
print(gauss_eliminasi(A, b, tampil_langkah=False))

## 11. Ringkasan Konsep

| Konsep | Arti sederhana |
|---|---|
| Matriks augmented | A dan b digabung dalam satu tabel |
| Pivot | Elemen diagonal yang dipakai untuk menghapus angka di bawahnya |
| Partial pivoting | Pilih pivot terbesar agar hitungan lebih akurat |
| Pengali `m` | Seberapa banyak baris pivot dikurangkan agar elemen menjadi 0 |
| `EPSILON` | Batas untuk menganggap angka "sama dengan nol" |
| Eliminasi maju | Mengubah matriks menjadi segitiga atas |
| Substitusi mundur | Menghitung variabel dari persamaan terbawah ke atas |

## 12. Latihan

1. Selesaikan SPL berikut dengan `gauss_eliminasi`, lalu verifikasi dengan `A @ x`:
   ```
   x + y + z = 6
   2y + 5z = -4
   2x + 5y - z = 27
   ```
2. Buat contoh SPL 2x2 yang tidak punya solusi (bukan hanya yang solusinya tak hingga), misalnya `x + y = 1` dan `x + y = 2`. Apa yang terjadi?
3. Coba matriks yang elemen diagonal awalnya 0, misalnya `A = [[0, 1], [1, 1]]`. Mengapa pivoting membuat program tetap bisa jalan?
4. Ubah ukuran menjadi 4x4 dan bandingkan hasilnya dengan `np.linalg.solve`.